# ARWO and separate entity accounts reconciliation
Checked 26 October 2026. Compare the four entities in 2023–2024. The 2025 source rows and ITC formula are inspected, but complete 2025 accounts have not been reconciled. Report values below are manually reviewed transcriptions; they are not an automated PDF extraction. See the linked research note for source pages and limitations.

Source lineage: preserved 2019–2023 CSV plus non-overlapping 2024/2025 extracts from ARWO_2019-2025.xlsx, Data_Raw_19-25. ITC comparator is gross XB delivery, not IPSAS non-RB expense. Differences use ARWO minus comparator, with the comparator as percentage denominator. No production data is changed.


In [1]:
from pathlib import Path
from decimal import Decimal
import pandas as pd
from openpyxl import load_workbook
root = Path.cwd() if Path('python').exists() else Path.cwd().parent
frames = [pd.read_csv(root / p, dtype=str) for p in [
    'data/un-secretariat-expenses.csv', 'data/un-secretariat-expenses-2024.csv',
    'data/un-secretariat-expenses-2025.csv']]
data = pd.concat(frames, ignore_index=True)
selected = data[(data.SOURCE_TYPE == 'Voluntary') & data.ENTITY.isin(['UNEP','UNODC','UN-HABITAT','ITC'])]
assert not selected.duplicated(['YEAR','ENTITY']).any()
# Reviewed transcription, USD. Reports print thousands; ITC gross XB is rounded millions.
# UNODC Note 18 separately verifies that regular-budget expenditure equals allocation.
reported = [
    (2023, 'UNEP', 696642000, 24701000, None),
    (2024, 'UNEP', 832075000, 24011000, None),
    (2023, 'UN-HABITAT', 162405000, 16578000, None),
    (2024, 'UN-HABITAT', 188264000, 15619000, None),
    (2023, 'UNODC', 451512000, 34657000, None),
    (2024, 'UNODC', 484798000, 37180000, None),
    (2023, 'ITC', None, None, 112000000),
    (2024, 'ITC', None, None, 117300000),
]
rows=[]
for year, entity, total, rb, gross in reported:
    row=selected[(selected.YEAR==str(year)) & (selected.ENTITY==entity)].iloc[0]
    arwo=Decimal(row.AMOUNT)
    comparable=Decimal(gross if gross is not None else total-rb)
    difference=arwo-comparable
    rows.append(dict(year=year, entity=entity, arwo_usd=str(arwo), report_comparator_usd=str(comparable),
        difference_usd=str(difference), difference_pct=float(difference/comparable*100),
        comparator='gross XB delivery (rounded)' if gross is not None else 'IPSAS expenses less audited RB expenses',
        arwo_reference=row.REFERENCE))
comparison=pd.DataFrame(rows)
print(comparison.to_string(index=False))
assert all(abs(Decimal(r['difference_usd'])) <= 1000 for r in rows if r['entity']!='ITC')
workbook=root/'data/internal/ARWO_2019-2025.xlsx'
formulas=load_workbook(workbook, data_only=False, read_only=True)
cached=load_workbook(workbook, data_only=True, read_only=True)
ws=formulas['Vol_ITC-25']; values=cached['Vol_ITC-25']
print('\nITC 2025 source-tab formula:', ws['I2'].value)
print('ITC 2025 source-tab cached value:', values['I2'].value)
print('ITC 2025 source-tab note:', ws['L2'].value)
print('Alternative calculation in L5:', ws['L5'].value, '=', values['L5'].value)
print('Difference between L5 and I2:', values['L5'].value-values['I2'].value)
assert ws['I2'].value == '=102991900/0.822'
assert ws['L5'].value == '=109410100/0.7737'
print('\n2025 ARWO rows (not independently reconciled to complete 2025 accounts):')
print(selected[selected.YEAR=='2025'][['ENTITY','AMOUNT','REFERENCE','NOTE']].to_string(index=False))


 year     entity    arwo_usd report_comparator_usd difference_usd  difference_pct                              comparator      arwo_reference
 2023       UNEP   671941000             671941000              0        0.000000 IPSAS expenses less audited RB expenses Financial Statement
 2024       UNEP 808064000.0             808064000            0.0        0.000000 IPSAS expenses less audited RB expenses Financial Statement
 2023 UN-HABITAT   145827000             145827000              0        0.000000 IPSAS expenses less audited RB expenses Financial Statement
 2024 UN-HABITAT 172645000.0             172645000            0.0        0.000000 IPSAS expenses less audited RB expenses Financial Statement
 2023      UNODC   416855704             416855000            704        0.000169 IPSAS expenses less audited RB expenses Financial Statement
 2024      UNODC 447617825.0             447618000         -175.0       -0.000039 IPSAS expenses less audited RB expenses Financial Statement
 2023 